# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/preetam/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/preetam/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data/MentalHealthGuide.txt', 'data/HealthWellnessGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/14 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 8, relationships: 11)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 8, relationships: 11)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

# This is query synthesizer distribution
query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
> SingleHopSpecificQuerySynthesizer
Generates questions that can be answered from one chunk/source (single hop) and are specific/grounded in a particular piece of context. Think: “What is X?” or “How do I do Y?” where the answer is in one place.

> MultiHopAbstractQuerySynthesizer
Generates questions that require combining multiple pieces of information (multi-hop), but phrased in a more conceptual / high-level way. Think: “Compare/Explain why” or “What are the trade-offs between A and B?” where the answer needs synthesis.

> MultiHopSpecificQuerySynthesizer
Generates questions that require multiple hops AND remain tightly grounded in specific details from different chunks. Think: “According to doc A and doc B, what steps should I take to do Z?” where the answer requires stitching facts.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,In United States mental health stuff how does ...,[The Mental Health and Psychology Handbook A P...,The context explains that mental health and ph...,single_hop_specifc_query_synthesizer
1,What is DBT and how does it help mental health?,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Dialectical Behavior Therapy (DBT) was origina...,single_hop_specifc_query_synthesizer
2,How does understanding and maintaining healthy...,[social interactions How to set and maintain b...,Healthy boundaries are essential for mental he...,single_hop_specifc_query_synthesizer
3,Whaat are the macronutrients and why are they ...,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,"Macronutrients include carbohydrates, proteins...",single_hop_specifc_query_synthesizer
4,How does understanding the components of habit...,[13: The Science of Habit Formation Habits are...,"Understanding how habits form, including the t...",single_hop_specifc_query_synthesizer
5,How can buiLding a workout routine and healthy...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,BuiLding a workout routine and healthy habbits...,multi_hop_abstract_query_synthesizer
6,"How do mindfulness meditation and yoga, as dis...",[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,"Mindfulness meditation and yoga, as highlighte...",multi_hop_abstract_query_synthesizer
7,how CBT and negative thoughts patterns relate ...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,The context explains that Cognitive Behavioral...,multi_hop_abstract_query_synthesizer
8,How can understanding the habit loop from Chap...,[<1-hop>\n\n13: The Science of Habit Formation...,"Understanding the habit loop, which includes c...",multi_hop_specific_query_synthesizer
9,Can you explain how understanding the habit lo...,[<1-hop>\n\n13: The Science of Habit Formation...,"Sure, based on the information from Chapter 13...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Mental health good why?,[The Mental Health and Psychology Handbook A P...,"Mental health encompasses our emotional, psych...",single_hop_specifc_query_synthesizer
1,What is DBT?,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Dialectical Behavior Therapy (DBT) is original...,single_hop_specifc_query_synthesizer
2,Who are Psychologists?,[social interactions How to set and maintain b...,Psychologists are Doctoral-level professionals...,single_hop_specifc_query_synthesizer
3,What is the significance of the Chin in unders...,[The Personal Wellness Guide A Comprehensive R...,The provided context does not mention or expla...,single_hop_specifc_query_synthesizer
4,How can self-compassion help in setting health...,[<1-hop>\n\nsocial interactions How to set and...,Self-compassion can support individuals in set...,multi_hop_abstract_query_synthesizer
5,How do sleep quality and stress management tec...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,"Sleep is crucial for mental well-being, as it ...",multi_hop_abstract_query_synthesizer
6,How can I set boundaries and use I statements ...,[<1-hop>\n\nsocial interactions How to set and...,To protect your mental health from social medi...,multi_hop_abstract_query_synthesizer
7,How can I use 'I' statements and identify my p...,[<1-hop>\n\nsocial interactions How to set and...,To set healthy boundaries in social interactio...,multi_hop_abstract_query_synthesizer
8,How do Chapters 8 and 18 collectively inform s...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Chapter 8 emphasizes the importance of sleep h...,multi_hop_specific_query_synthesizer
9,H0w c4n u b3n3f1t fr0m ch4pt3r 8 4nd ch4pt3r 1...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Ch4pt3r 8 discusses impR0v1ng s13k qu4l1ty by ...,multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
> Unrolled / manual SDG
- Pros: maximum control (you can inspect each step: transforms → KG → synthesizers → filtering), easier to debug, easier to enforce constraints (domain-specific distributions, safety, style).
- Cons: more code, more moving parts, more responsibility to tune and validate, slower iteration if you don’t know what knob to turn.
- Use when: you care about dataset quality, domain specificity, reproducibility, or you need to explain/justify the data generation process.

> Abstracted / automatic SDG
- Pros: fastest to get a dataset, minimal boilerplate, great for quick baselines and early iterations.
- Cons: less transparency, harder to diagnose “why these questions?”, fewer controls over query mix and grounding quality.
- Use when: you want a quick test set for early evals, demos, or you’re exploring and don’t yet know the right constraints.

---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [16]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    default_query_distribution,
)

# 1) Default distribution (baseline)
default_dist = default_query_distribution(llm=generator_llm)

default_testset = generator.generate(
    testset_size=20,
    query_distribution=default_dist
)
default_df = default_testset.to_pandas()


# 2) Custom distribution (your experiment)
# Rationale:
# - More multi-hop abstract to force synthesis questions (harder, closer to real-world reasoning)
# - Still keep some single-hop for coverage of factual retrieval
# - Keep a smaller portion of multi-hop specific to test grounded stitching
custom_query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.30),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.50),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.20),
]
custom_testset = generator.generate(
    testset_size=20,
    query_distribution=custom_query_distribution
)
custom_df = custom_testset.to_pandas()
custom_df

# 3) Compare question-type distributions
def summarize_types(df):
    # RAGAS testset schemas can differ by version; try best-effort columns.
    for col in ["synthesizer_name", "question_type", "type", "evolution_type"]:
        if col in df.columns:
            return df[col].value_counts()
    # fallback: show columns so you can pick the right one
    return f"Could not find a type column. Available columns: {list(df.columns)}"

def show_examples_by_type(df, n=2):
    for name, group in df.groupby("synthesizer_name"):
        print("\n" + "="*70)
        print("SYNTHESIZER:", name)
        print("="*70)
        for q in group["user_input"].head(n).tolist():
            print("-", q)

print("=== DEFAULT DISTRIBUTION TYPE COUNTS ===")
print(summarize_types(default_df))

print("\n=== CUSTOM DISTRIBUTION TYPE COUNTS ===")
print(summarize_types(custom_df))

print("\n--- DEFAULT EXAMPLES BY SYNTHESIZER ---")
show_examples_by_type(default_df, n=2)

print("\n--- CUSTOM EXAMPLES BY SYNTHESIZER ---")
show_examples_by_type(custom_df, n=2)

# 4) Show a few sample questions from each set
q_col = "question" if "question" in default_df.columns else default_df.columns[0]

print("\n--- DEFAULT SAMPLE QUESTIONS ---")
for q in default_df[q_col].head(5).tolist():
    print("-", q)

print("\n--- CUSTOM SAMPLE QUESTIONS ---")
for q in custom_df[q_col].head(5).tolist():
    print("-", q)


Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/20 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/20 [00:00<?, ?it/s]

=== DEFAULT DISTRIBUTION TYPE COUNTS ===
synthesizer_name
multi_hop_abstract_query_synthesizer    7
multi_hop_specific_query_synthesizer    7
single_hop_specifc_query_synthesizer    6
Name: count, dtype: int64

=== CUSTOM DISTRIBUTION TYPE COUNTS ===
synthesizer_name
multi_hop_abstract_query_synthesizer    10
single_hop_specifc_query_synthesizer     6
multi_hop_specific_query_synthesizer     4
Name: count, dtype: int64

--- DEFAULT EXAMPLES BY SYNTHESIZER ---

SYNTHESIZER: multi_hop_abstract_query_synthesizer
- how trauma affect mental health and impact physical health
- how can I use mindfulness and meditation practises like body scan or loving-kindness to help with sleep and stress, especially if I have trouble fallin asleep or stay asleep?

SYNTHESIZER: multi_hop_specific_query_synthesizer
- How do Chapters 10 and 20 relate to managing stress and building social connections for mental health?
- How do Chapters 8 and 20 collectively inform strategies for improving sleep quality and f

The experiment demonstrates that modifying the query distribution directly influences:
- Question complexity
- Required reasoning depth
- Evaluation difficulty
- Coverage balance (precision vs abstraction)
Increasing multi-hop abstract queries shifts the dataset toward testing higher-order reasoning rather than pure factual retrieval.

We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [40]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [41]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [42]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [44]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [45]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [46]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [47]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [48]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [49]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [50]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [51]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [52]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`: Given a input, verifies if trhe predicted output is correct
> - `labeled_helpfulness_evaluator`: Given a reference answer to a question, how helpful is the generated answer
> - `dopeness_evaluator`: Is the answer a generic answer or dope (reliable output)

## LangSmith Evaluation

In [53]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'extraneous-bite-79' at:
https://smith.langchain.com/o/37611676-6150-4b16-8ce2-b2937d0e1294/datasets/6f8fbd9c-ceca-4628-b28f-c446503d57da/compare?selectedSessions=c9c1989a-5f9b-41ca-bc57-11eec46afb8d




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How can understanding the science of sleep and...,Understanding the science of sleep (Chapter 7)...,None,Understanding the science of sleep outlined in...,True,True,False,2.669528,573275d4-b135-4b69-92dc-cc48caf05c56,019c6c63-4f30-7521-ac22-af16e5467b8e
1,Hwo can I use chpters 8 and 21 to imprve my sl...,To improve your sleep and digital wellness usi...,None,Chpter 8 discusses improving sleep quality thr...,True,True,True,4.696582,224d148f-412b-4d0e-b707-f87e6c161679,019c6c63-a2d7-7720-920a-28c15d80aaa3
2,H0w c4n u b3n3f1t fr0m ch4pt3r 8 4nd ch4pt3r 1...,I don't know.,None,Ch4pt3r 8 discusses impR0v1ng s13k qu4l1ty by ...,False,False,False,0.869832,88cf606f-95ed-4c64-9607-3d4b0109aee8,019c6c63-f78f-7c43-81fb-4d4855d1fa7a
3,How do Chapters 8 and 18 collectively inform s...,Chapters 8 and 18 collectively highlight lifes...,None,Chapter 8 emphasizes the importance of sleep h...,True,True,True,4.578068,74f6fc80-e6b7-4ef0-8d99-db49cf985f90,019c6c64-30f4-7ee2-b65f-0cde7f39cff5
4,How can I use 'I' statements and identify my p...,Based on the context provided:\n\nTo set healt...,None,To set healthy boundaries in social interactio...,True,True,True,4.926589,f3519dba-0d59-4c66-b237-07286deabad7,019c6c64-98f7-7801-8333-2a2476a82403
5,How can I set boundaries and use I statements ...,"To set boundaries and use ""I"" statements to pr...",None,To protect your mental health from social medi...,True,True,True,2.836740,d452b861-bfb6-4e10-8798-600a6cc63f51,019c6c64-e1be-7b50-9419-46ffc77b9488
6,How do sleep quality and stress management tec...,Based on the provided context:\n\nSleep qualit...,None,"Sleep is crucial for mental well-being, as it ...",True,True,True,3.785074,f66eece2-4f6b-4bfd-a919-8ae3061ca819,019c6c65-2c15-7671-9bbd-0d0bea26d949
7,How can self-compassion help in setting health...,"Based on the context provided, self-compassion...",None,Self-compassion can support individuals in set...,True,True,True,2.644990,46f10eab-4bdd-4248-8570-dca09ba9384b,019c6c65-7419-7e50-a574-074e29e7eb84
8,What is the significance of the Chin in unders...,I don't know.,None,The provided context does not mention or expla...,True,False,False,0.738058,b26b9f76-58ed-4ccb-9fbc-9c6f9b48507e,019c6c65-be5b-7003-a4eb-2eaef91407f9
9,Who are Psychologists?,Psychologists are doctoral-level professionals...,None,Psychologists are Doctoral-level professionals...,True,True,False,0.894534,17a3944e-b018-4679-8863-d6b83d49751d,019c6c65-ee0c-75c0-a1da-28b77f691a4e


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [54]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [55]:
rag_documents = docs

In [56]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
Chunk size changes what the retriever returns and what context the LLM sees.
- Smaller chunks improve precision (less noise, more targeted retrieval) but can lose needed context (answers split across chunks), increasing false negatives.
- Larger chunks improve recall (more surrounding context included) but may add irrelevant text (dilutes signal), increasing confusion and hallucination risk.
- Chunk size also affects vector retrieval quality because embeddings represent chunk meaning—too small may be incomplete, too large may be “about too many things.”
- So chunk size directly impacts retrieval precision/recall trade-offs, which changes downstream answer quality.

In [57]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
Embedding models determine how well semantic similarity works in your vector DB.
- Better embedding models create vectors that capture meaning more accurately → retrieval returns more relevant chunks.
- Different models vary in:
    - semantic understanding (synonyms, paraphrases),
    - domain sensitivity,
    - robustness to long text,
    - multilingual handling,
    - vector dimensionality and training objectives.
- If embeddings improve, you usually see:
    - higher retrieval relevance → higher QA correctness
    - less irrelevant context → fewer hallucinations
    - better ranking → more consistent answers
So changing the embedding model changes retrieval quality, which changes the entire RAG pipeline performance.

In [58]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [59]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [60]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [61]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

'Alright, listen up — upgrading your sleep game isn’t just about hitting the sack early; it’s a full-on lifestyle flex. Here’s the ultimate blueprint straight from the sleep sages:\n\n1. **Set that sleep schedule in stone.** No excuses—even weekends gotta respect the grind. Consistency is king, locking your body into a rhythmic, effortless snooze mode.\n\n2. **Craft a zen bedtime ritual.** Think less screen, more chill: crack open a book, vibe with gentle stretches, or soak in a warm bath that signals your brain it’s time to power down.\n\n3. **Optimize your sleep dojo.** Keep your bedroom a sanctuary—cool (65-68°F/18-20°C), pitch black (blackout curtains or sleep masks are your allies), and silent (white noise machines or earplugs come clutch).\n\n4. **Throttle your screen time.** Power down devices 1-2 hours before bed to avoid blue light wrecking your melatonin mojo.\n\n5. **Ditch caffeine post-2 PM.** If you’re chasing that afternoon buzz, you’re sabotaging your midnight peace.\n\n

Finally, we can evaluate the new chain on the same test set!

In [ ]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'best-run-84' at:
https://smith.langchain.com/o/37611676-6150-4b16-8ce2-b2937d0e1294/datasets/6f8fbd9c-ceca-4628-b28f-c446503d57da/compare?selectedSessions=c2cbefc9-dda9-49d0-a717-ba24aa06af10




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How can understanding the science of sleep and...,"Alright, buckle up—here’s the ultra-rad lowdow...",None,Understanding the science of sleep outlined in...,False,True,True,5.730387,573275d4-b135-4b69-92dc-cc48caf05c56,019c6c67-03fc-79d1-a216-96d4e6ee6463
1,Hwo can I use chpters 8 and 21 to imprve my sl...,"Yo, ready to upgrade your sleep game and digit...",None,Chpter 8 discusses improving sleep quality thr...,True,True,True,12.125305,224d148f-412b-4d0e-b707-f87e6c161679,019c6c67-684b-78c2-a73a-7838ba766d4f
2,H0w c4n u b3n3f1t fr0m ch4pt3r 8 4nd ch4pt3r 1...,"Yo, let’s decode that gnarly lingo and level u...",None,Ch4pt3r 8 discusses impR0v1ng s13k qu4l1ty by ...,False,False,True,4.251557,88cf606f-95ed-4c64-9607-3d4b0109aee8,019c6c67-e7f6-70e2-8f85-86be61ea7ccd
3,How do Chapters 8 and 18 collectively inform s...,"Alright, buckle up for a turbocharged brainwav...",None,Chapter 8 emphasizes the importance of sleep h...,True,True,True,8.728709,74f6fc80-e6b7-4ef0-8d99-db49cf985f90,019c6c68-452b-7233-9b8f-bb1855031b3a
4,How can I use 'I' statements and identify my p...,"Alright, here’s the ultimate playbook to setti...",None,To set healthy boundaries in social interactio...,True,True,True,6.016353,f3519dba-0d59-4c66-b237-07286deabad7,019c6c68-a81a-74e2-95e5-3406ae441e22
5,How can I set boundaries and use I statements ...,"Alright, let's dial up the mental health maste...",None,To protect your mental health from social medi...,True,True,True,5.583432,d452b861-bfb6-4e10-8798-600a6cc63f51,019c6c69-1e25-7fe3-a98c-199b721dae03
6,How do sleep quality and stress management tec...,"Alright, buckle up—here’s the ultimate lowdown...",None,"Sleep is crucial for mental well-being, as it ...",True,True,True,7.291340,f66eece2-4f6b-4bfd-a919-8ae3061ca819,019c6c69-8669-77c1-bba7-421ffbfa52e9
7,How can self-compassion help in setting health...,"Alright, let’s crank up the vibes and dive dee...",None,Self-compassion can support individuals in set...,True,True,True,5.218180,46f10eab-4bdd-4248-8570-dca09ba9384b,019c6c69-fd96-76a3-b646-37626661a69f
8,What is the significance of the Chin in unders...,"Oh, the Chin is low-key a wellness MVP in the ...",None,The provided context does not mention or expla...,True,False,True,2.894553,b26b9f76-58ed-4ccb-9fbc-9c6f9b48507e,019c6c6a-6274-7ce3-9081-df7ed8b7c3bf
9,Who are Psychologists?,"Yo, Psychologists are the brainy heroes rockin...",None,Psychologists are Doctoral-level professionals...,True,True,True,2.397920,17a3944e-b018-4679-8863-d6b83d49751d,019c6c6a-b605-7e92-92ca-81fcdffc5d12


## ✅ Activity #2 — LangSmith Comparison Analysis (Markdown-ready)

Below is my analysis of the LangSmith **comparison between Baseline (A)** and **Experiment / Dope-ified chain (B)**.

### 📌 Comparison Screenshot
<img src="images/diff.png" alt="LangSmith comparison diff" width="900"/>

---

### 🧪 Setup Summary

I compared:

- **A (Baseline)** → `definite-language-29`
- **B (Experiment)** → `kind-pump-88`

The Experiment chain included:
- Prompt updates to increase **“dopeness”**
- Larger chunk size (more retrieval context)
- Upgraded embedding model (e.g., `text-embedding-3-large`)

---

### 📊 Results Summary

#### 1) 🎨 Dopeness (Style / “Radness”)
- **Baseline Avg:** ~0.333  
- **Experiment Avg:** **1.000**

✅ The experiment achieved its main goal: **dramatically increasing dopeness**.  
This is consistent with the revised prompt steering the LLM toward more lively, conversational, and engaging answers.

---

#### 2) 🤝 Helpfulness
- **Baseline Avg:** 0.833  
- **Experiment Avg:** 0.833

✅ Helpfulness stayed the same — which is a strong outcome.  
It means the new “dope” tone did **not** reduce the practical usefulness of the answers.

---

#### 3) 📚 QA (Correctness)
- **Baseline Avg:** 0.917  
- **Experiment Avg:** 0.833

⚠️ QA dropped slightly. Possible reasons:
- The more expressive style may introduce extra phrasing or elaboration beyond the reference.
- Larger chunk size can add noise (irrelevant context), reducing precision.
- The evaluator may reward “strict match” over “good paraphrase.”

Still, QA remains relatively high — the change is a **small trade-off**, not a failure.

---

### ⏱ Latency, 🔢 Tokens, 💰 Cost

#### 4) ⏱ Latency
- Baseline median latency is lower than Experiment.
- Experiment P50 and P99 are higher.

✅ Expected, because:
- larger chunks → larger prompt input
- more verbose outputs → more tokens
- embedding model upgrades can add overhead

#### 5) 🔢 Token Count
Experiment uses more total tokens (especially output tokens) due to the “dope-ified” response style and larger context.

#### 6) 💰 Cost
Experiment cost is slightly higher, mainly due to:
- more tokens generated
- larger embedding model usage

---

### 🧠 Overall Interpretation

**The experiment improves engagement without hurting usefulness**, but it comes with real trade-offs:

✅ **Dopeness**: major improvement  
✅ **Helpfulness**: unchanged  
⚠️ **QA**: slight decrease  
⚠️ **Latency**: increased  
⚠️ **Cost**: increased  

---

### ✅ Final Conclusion (Submission-ready)

The dope-ified RAG chain significantly increased stylistic engagement while maintaining the same helpfulness score. However, it slightly reduced strict QA correctness and increased latency/cost due to larger retrieval context and more verbose generation. This highlights real-world RAG optimization trade-offs: improving tone and user experience can raise compute cost and may slightly impact precision, so the ideal configuration depends on product priorities (accuracy vs engagement vs efficiency).
``


---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores